# 04 - LSTM / GRU

Task 3.3: windowed LSTM/GRU on scaled returns (Listing 3.3), Optuna hyperparameter tuning, and 5-seed reporting. Requires `tensorflow` (optional, heavy dependency - `pip install tensorflow`).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config

In [ ]:
try:
    from src.models import dl_models
    TF_OK = dl_models.TF_AVAILABLE
except ImportError:
    TF_OK = False
print("TensorFlow available:", TF_OK)

In [ ]:
df = pd.read_parquet(config.PROCESSED_DATA_DIR / "prices_clean.parquet")
train = pd.read_parquet(config.PROCESSED_DATA_DIR / "train.parquet")
val = pd.read_parquet(config.PROCESSED_DATA_DIR / "val.parquet")
test = pd.read_parquet(config.PROCESSED_DATA_DIR / "test.parquet")

train_ret = df.loc[train.index.min():train.index.max(), "ret"]
val_ret = df.loc[val.index.min():val.index.max(), "ret"]

In [ ]:
if TF_OK:
    best_params, study = dl_models.tune_hyperparameters(train_ret, val_ret, n_trials=10)
    print(best_params)

In [ ]:
if TF_OK:
    models, scaler, seed_summary = dl_models.run_multi_seed(train_ret, val_ret, best_params)
    print(f"val RMSE: {seed_summary['mean_val_rmse']:.6f} +/- {seed_summary['std_val_rmse']:.6f}")

In [ ]:
if TF_OK:
    from src import features
    full_ret = df["ret"]
    chosen = models[0]
    ret_preds = dl_models.walk_forward_predict_dl(chosen, scaler, full_ret, test.index[:60], best_params["lookback"])
    price_preds = features.reconstruct_price_from_returns(df["log_close"], ret_preds)
    price_preds = features.shift_to_target_dates(price_preds, df.index)
    ax = df.loc[price_preds.index, "Adj Close"].plot(label="Actual", figsize=(11, 4))
    price_preds.plot(ax=ax, label="LSTM/GRU")
    ax.legend(); plt.show()